# 🏎️ LazyPredict Benchmarking
**Extended · Data Pattern Recognition for the Rest of Us**

> Before committing to a model, benchmark dozens of algorithms in one call. LazyPredict tells you which model families are worth deeper investment — in minutes instead of days.

### Dataset
**Telecom Customer Churn**
Predict which customers will cancel their service. Churn costs telecom companies billions annually — retaining customers is 5-25x cheaper than acquiring new ones. This is a classic imbalanced binary classification problem with real business stakes.

### Time: ~45 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
!pip install lazypredict -q
from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
print("\u2713 Setup complete — lazypredict installed")

In [ ]:
# Telecom Churn Dataset (IBM Telco — widely used in industry)
import numpy as np, pandas as pd

try:
    url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
    telco = pd.read_csv(url)
    telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce')
    telco = telco.dropna()
    telco['Churn_num'] = (telco['Churn'] == 'Yes').astype(int)
    cat_cols = telco.select_dtypes('object').columns.drop(['customerID','Churn'])
    le = LabelEncoder()
    for c in cat_cols:
        telco[c] = le.fit_transform(telco[c])
    feature_cols = [c for c in telco.columns
                    if c not in ['customerID','Churn','Churn_num']]
    X = telco[feature_cols].values
    y = telco['Churn_num'].values
    print("Loaded IBM Telco Churn dataset")
except:
    np.random.seed(42)
    n = 7043  # matches real Telco dataset size
    tenure         = np.random.randint(0, 72, n)
    monthly_charges= np.random.uniform(18, 120, n)
    total_charges  = tenure * monthly_charges * np.random.uniform(0.9,1.1,n)
    contract       = np.random.choice([0,1,2], n, p=[0.55,0.24,0.21])  # M2M/1yr/2yr
    internet_service=np.random.choice([0,1,2], n, p=[0.21,0.44,0.35])  # No/DSL/Fiber
    tech_support   = np.random.binomial(1,0.28,n)
    paperless      = np.random.binomial(1,0.59,n)
    num_services   = np.random.randint(0,7,n)
    senior         = np.random.binomial(1,0.16,n)

    log_odds = (-1.5
                - 0.05*tenure
                + 0.015*monthly_charges
                + 1.2*(contract==0).astype(float)
                - 0.8*(contract==2).astype(float)
                + 0.4*(internet_service==2).astype(float)
                - 0.3*tech_support
                + 0.2*senior)
    prob_churn = 1/(1+np.exp(-log_odds))
    churn = (np.random.uniform(0,1,n) < prob_churn).astype(int)

    X = np.column_stack([tenure,monthly_charges,total_charges,contract,
                         internet_service,tech_support,paperless,
                         num_services,senior])
    y = churn
    feature_cols = ['tenure','monthly_charges','total_charges','contract',
                    'internet_service','tech_support','paperless',
                    'num_services','senior']
    print("Using synthetic Telco-like churn dataset")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                             random_state=42, stratify=y)
print(f"\nTelecom Churn dataset: {len(y):,} customers")
print(f"Churn rate: {y.mean():.1%}")
print(f"Training: {len(y_tr):,}  |  Test: {len(y_te):,}")
print("\nRunning LazyPredict on all sklearn classifiers...")
print("(This benchmarks 30+ models — may take 2-3 minutes)")

In [ ]:
# Run LazyPredict — benchmark all classifiers
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models_df, predictions = clf.fit(X_tr, X_te, y_tr, y_te)

print("=== LazyPredict Results — Telecom Churn (sorted by AUC-ROC) ===\n")
# Show top 15
display_cols = [c for c in ['Accuracy','Balanced Accuracy','ROC AUC','F1 Score','Time Taken']
                if c in models_df.columns]
if 'ROC AUC' in models_df.columns:
    models_sorted = models_df.sort_values('ROC AUC', ascending=False)
else:
    models_sorted = models_df.sort_values('Accuracy', ascending=False)
print(models_sorted.head(15)[display_cols].to_string())

In [ ]:
# Visualize the benchmark results
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# AUC-ROC comparison
metric = 'ROC AUC' if 'ROC AUC' in models_df.columns else 'Accuracy'
top_models = models_sorted.head(15)
colors_bar = ['#1a7a45' if i < 3 else '#1e5fa8' if i < 8 else '#888'
              for i in range(len(top_models))]
axes[0].barh(range(len(top_models)), top_models[metric].values[::-1],
             color=colors_bar[::-1], edgecolor='white')
axes[0].set_yticks(range(len(top_models)))
axes[0].set_yticklabels([n[:25] for n in top_models.index[::-1]], fontsize=9)
axes[0].set_xlabel(metric)
axes[0].set_title(f'LazyPredict: {metric} by Model\n'
                  f'Green = top 3, Blue = top 8')
axes[0].axvline(top_models[metric].median(), color='#e85d20', lw=1.5, ls='--',
               label=f'Median = {top_models[metric].median():.3f}')
axes[0].legend()

# Speed vs accuracy tradeoff
if 'Time Taken' in models_df.columns:
    axes[1].scatter(models_df['Time Taken'], models_df[metric],
                    color='#1e5fa8', alpha=0.7, s=40)
    for i, (name, row) in enumerate(models_df.iterrows()):
        if row[metric] > models_df[metric].quantile(0.85):
            axes[1].annotate(name[:15], (row['Time Taken'], row[metric]),
                            fontsize=7, alpha=0.8)
    axes[1].set_xlabel('Training Time (seconds)')
    axes[1].set_ylabel(metric)
    axes[1].set_title('Speed vs Performance Tradeoff\n'
                      'Best = top-left (fast AND accurate)')

plt.tight_layout(); plt.show()
best_model = models_sorted.index[0]
print(f"\n\u2714 LazyPredict winner: {best_model}")
print(f"   This is a STARTING POINT — now tune this model with proper cross-validation")
print(f"   LazyPredict does NOT replace rigorous model evaluation!")

In [ ]:
# Business framing: what does AUC mean for churn?
print("=== Business Interpretation ===\n")
print("A churn model with AUC=0.85 means:")
print("  If you randomly pick 1 churner and 1 non-churner,")
print("  the model ranks the churner higher 85% of the time.")
print()
print("At a retention budget of $50/customer and 7,043 customers:")
churn_rate = y.mean()
n_churners = int(len(y) * churn_rate)
budget_per_customer = 50
print(f"  Total churners to prevent: ~{n_churners}")
print(f"  If model catches 70% of churners at a 2:1 false positive rate:")
caught = int(n_churners * 0.70)
contacted = caught * 3  # 2:1 FP ratio
cost = contacted * budget_per_customer
prevented_revenue = caught * 120 * 12  # $120/mo avg, 1 year
print(f"  Customers contacted: {contacted:,}")
print(f"  Campaign cost: ${cost:,}")
print(f"  Revenue retained (est.): ${prevented_revenue:,}")
print(f"  Net ROI: {(prevented_revenue - cost)/cost*100:.0f}%")

In [ ]:
# @title 📝 Quiz — LazyPredict Benchmarking
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is LazyPredict's primary use case and what is it NOT good for?
# @markdown **Q2:** Why is accuracy a poor metric for churn prediction (hint: class imbalance)?
# @markdown **Q3:** The top LazyPredict model uses default hyperparameters. What should you do next?
# @markdown **Q4:** How would you explain AUC-ROC to a non-technical marketing manager?
# @markdown **Q5:** For churn: would you rather minimize false positives or false negatives? Why?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "LazyPredict Benchmarking"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*